# PCA-Clustering Baseline (Supplementary / Rebuttal Experiment)

> **What this is.** A supplementary baseline added in response to a reviewer request. The reviewer asked whether the partition structure our method discovers is specific to the **interchangeability-graph** framing, or whether it could be recovered by a simpler *diagnostic* approach such as clustering on residual-stream activations. This notebook answers that by replacing **Step 2** of our pipeline (interchange graph + quasi-clique partition) with a plain **PCA + K-means** partition on raw activations, and comparing the two on identical interchange graphs.
>
> It is **not** part of the core method — it is an external point of comparison that quantifies how much of the discovered structure is attributable to our framing.

Consolidated across all four configurations reported in the paper:

| # | Experiment | Model | Layer/Pos | K |
|---|---|---|---|---|
| 1 | Logic `o4` @ L5/P78 | fine-tuned GPT-2 small | L5, P78 (op4) | 2 |
| 2 | Logic `o5` @ L7/P77 | fine-tuned GPT-2 small | L7, P77 (op5) | 2 |
| 3 | Entity Binding @ L15 | google/gemma-2-2b-it | L15, last token | 2 |
| 4 | RAVEL Language @ L14 | meta-llama/Llama-3.1-8B (cached) | L14, city last token | 2 |

For each row we (a) extract residual-stream activations, (b) reduce with PCA (50 dims), (c) cluster with K-means into the paper's `K`, (d) compute per-bucket IIA on the paper's adjacency matrix, and (e) compare against the quasi-clique partition (recomputed at the paper's stated `γ = 0.98`) via **ARI / NMI**. A summary table at the bottom puts our-method vs PCA-baseline numbers side by side.

**How to read the result.** Low ARI/NMI + a clearly higher target-bucket IIA for our method ⇒ the interchangeability-graph framing captures structure that activation geometry alone does **not** recover (supports the method). High ARI/NMI ⇒ PCA already recovers the same partition (would weaken the framing's added value).

**Runtime.** Colab GPU runtime (T4 or better) recommended. RAVEL uses cached activations and needs no model load; the two Logic configs and Entity Binding each load their model once.

## 1. Setup

Clones the project repo and installs dependencies. We deliberately **do not** install the `causalab` submodule — none of the activation extraction we do actually needs it (the entity-binding code below reconstructs prompts directly from saved samples). This avoids a dependency on `causalab.tasks.entity_binding`, which lives only in the author's internal fork.

In [ ]:
import os, sys, subprocess, shutil

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    REPO_ROOT = '/content/apple-bucket'
    if not os.path.isdir(REPO_ROOT):
        !git clone https://github.com/Paulineli/apple-bucket.git {REPO_ROOT}
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

assert os.path.isdir(REPO_ROOT), f'REPO_ROOT not found: {REPO_ROOT}'
print('REPO_ROOT =', REPO_ROOT)

In [ ]:
!pip install -q torch transformers accelerate scikit-learn tqdm numpy pandas pyvene huggingface_hub

In [ ]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()  # paste an HF token with access to google/gemma-2-2b-it

### 1.1 Download fine-tuned GPT-2 weights from Hugging Face

The fine-tuned GPT-2 small checkpoint (Boolean-logic task) is hosted on the Hugging Face Hub at [`PaulineLi/bucketing-good-apples-gpt2-logic`](https://huggingface.co/PaulineLi/bucketing-good-apples-gpt2-logic). The cell below fetches it into `experiments/logic_task/artifacts/models/fine_tuned_gpt2_or/` so that `util_model.load_model()` finds a complete local model directory (weights + config + tokenizer).

The download is skipped if a valid directory is already in place. The repo is public, so no token is needed. If the download is unavailable, `util_model.load_model()` falls back to vanilla `gpt2`, in which case the Logic activations will not match the paper's.

In [ ]:
from huggingface_hub import snapshot_download

GPT2_HF_REPO       = 'PaulineLi/bucketing-good-apples-gpt2-logic'
GPT2_MODELS_PARENT = os.path.join(REPO_ROOT, 'experiments', 'logic_task', 'artifacts', 'models')
GPT2_MODEL_DIR     = os.path.join(GPT2_MODELS_PARENT, 'fine_tuned_gpt2_or')

def _is_valid_gpt2_dir(d):
    return os.path.isdir(d) and os.path.exists(os.path.join(d, 'config.json')) and (
        os.path.exists(os.path.join(d, 'pytorch_model.bin')) or
        os.path.exists(os.path.join(d, 'model.safetensors'))
    )

if _is_valid_gpt2_dir(GPT2_MODEL_DIR):
    print('Fine-tuned GPT-2 already present at', GPT2_MODEL_DIR)
else:
    os.makedirs(GPT2_MODEL_DIR, exist_ok=True)
    print(f'Downloading {GPT2_HF_REPO} -> {GPT2_MODEL_DIR} ...')
    snapshot_download(
        repo_id=GPT2_HF_REPO,
        repo_type='model',
        local_dir=GPT2_MODEL_DIR,
        # only the files needed to load the model
        allow_patterns=['*.safetensors', '*.bin', 'config.json',
                        'tokenizer*.json', 'vocab.json', 'merges.txt',
                        'special_tokens_map.json'],
    )
    assert _is_valid_gpt2_dir(GPT2_MODEL_DIR), (
        f'Download did not produce a valid model dir at {GPT2_MODEL_DIR}: '
        f'{sorted(os.listdir(GPT2_MODEL_DIR))}'
    )
    print('Ready:', GPT2_MODEL_DIR, '->', sorted(os.listdir(GPT2_MODEL_DIR)))

## 2. Common utilities and config

Shared metrics, sys.path bootstrapping for each experiment's scripts, and the dict of per-experiment hyperparameters / file paths.

In [ ]:
import json, pickle
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

SEED              = 42
N_PCA_COMPONENTS  = 50
GAMMA             = 0.98   # density threshold for quasi-clique (paper §3 and Appendix: "We choose γ = 0.98 for all experiments")
MIN_CLIQUE_SIZE   = 2
DEVICE            = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| gamma =', GAMMA)

for sub in ('logic_task', 'entity_binding', 'factual_recall'):
    p = os.path.join(REPO_ROOT, 'experiments', sub, 'scripts')
    if p not in sys.path:
        sys.path.insert(0, p)

# Import the paper's own quasi-clique partitioner so we use the same algorithm,
# just with the paper's stated gamma instead of whatever happened to be saved in the JSONs.
from partition_graph_quasi_clique import quasi_clique_partition

In [ ]:
def edge_density(adj: np.ndarray) -> float:
    n = adj.shape[0]
    if n <= 1:
        return 0.0
    return float(int(adj.sum()) // 2) / (n * (n - 1) // 2)

def per_bucket_iia(adj: np.ndarray, labels: np.ndarray, k: int):
    out = {}
    for c in range(k):
        mask = labels == c
        sub = adj[mask][:, mask]
        out[int(c)] = {'size': int(mask.sum()), 'iia': edge_density(sub)}
    return out

def weighted_mean_iia(per_bucket):
    total = sum(info['size'] for info in per_bucket.values())
    return sum(info['size'] * info['iia'] for info in per_bucket.values()) / max(total, 1)

def pca_kmeans_cluster(X: np.ndarray, k: int, n_components: int, seed: int):
    n_comp = min(n_components, X.shape[0], X.shape[1])
    pca = PCA(n_components=n_comp, random_state=seed)
    X_pca = pca.fit_transform(X)
    km = KMeans(n_clusters=k, n_init=20, random_state=seed)
    labels = km.fit_predict(X_pca)
    return labels, float(pca.explained_variance_ratio_.sum()), n_comp

def evaluate_partition(adj, labels, k):
    pb = per_bucket_iia(adj, labels, k)
    return {'labels': labels.tolist(),
            'per_bucket': pb,
            'weighted_mean_iia': weighted_mean_iia(pb)}

## 3. Experiment 1+2 — Logic task (L5/P78 and L7/P77)

Both Logic configs share the same fine-tuned GPT-2 small model and same activation-extraction helper (`extract_activations_at_layer_pos` from `step4_analyze.py`); only the `(layer, pos, intervention, graph file, partition file)` tuple differs. We load the model once, then run both configurations.

In [ ]:
import util_data, util_model
from step4_analyze import extract_activations_at_layer_pos

print('Loading fine-tuned GPT-2 small ...')
_gpt2_model, _gpt2_tok = util_model.load_model()
_gpt2_model = _gpt2_model.to(DEVICE).eval()
print('  loaded')

In [ ]:
LOGIC_ART = os.path.join(REPO_ROOT, 'experiments', 'logic_task', 'artifacts', 'partition_results_das')

LOGIC_CONFIGS = [
    {
        'name': 'Logic o4 @ L5/P78',
        'layer': 5, 'pos': 78, 'K': 2,
        'graph': os.path.join(LOGIC_ART, 'graph_das_L5_P78_200_op4_with_directed.pkl'),
        'dataset': os.path.join(LOGIC_ART, 'graph_dataset_200_op4.pkl'),
        'saved_partition': os.path.join(LOGIC_ART, 'partition_results_das_L5_P78_K2_200_op4.json'),
    },
    {
        'name': 'Logic o5 @ L7/P77',
        'layer': 7, 'pos': 77, 'K': 2,
        'graph': os.path.join(LOGIC_ART, 'graph_das_L7_P77_200_op5_with_directed.pkl'),
        'dataset': os.path.join(LOGIC_ART, 'graph_dataset_200_op5_ds.pkl'),
        'saved_partition': os.path.join(LOGIC_ART, 'partition_results_das_L7_P77_K2_200_op5.json'),
    },
]

results = {}

for cfg in LOGIC_CONFIGS:
    print(f"\n=== {cfg['name']} ===")
    with open(cfg['graph'], 'rb') as f:
        adj = np.asarray(pickle.load(f)['undirected'], dtype=bool)
    with open(cfg['dataset'], 'rb') as f:
        ds = pickle.load(f)
    assert adj.shape[0] == len(ds)
    n = adj.shape[0]

    # Show what gamma the saved JSON used, for transparency.
    with open(cfg['saved_partition']) as f:
        _saved = json.load(f)
    print(f"  Saved JSON used gamma={_saved.get('gamma')}; recomputing with gamma={GAMMA} to match paper.")

    cache = os.path.join(LOGIC_ART, f"activations_L{cfg['layer']}_P{cfg['pos']}_n{n}.pt")
    if os.path.exists(cache):
        print(f'  Loading cached activations from {cache}')
        acts = torch.load(cache, map_location='cpu')
    else:
        acts = extract_activations_at_layer_pos(
            _gpt2_model, _gpt2_tok, ds,
            layer=cfg['layer'], pos_num=cfg['pos'],
            device=DEVICE, batch_size=32,
        )
        torch.save(acts, cache)
    X = acts.detach().cpu().numpy().astype(np.float32)

    pca_lab, evr, n_comp = pca_kmeans_cluster(X, cfg['K'], N_PCA_COMPONENTS, SEED)
    # Recompute paper labels live with the paper's stated gamma=0.98.
    paper_labels = np.asarray(quasi_clique_partition(adj, cfg['K'], GAMMA, MIN_CLIQUE_SIZE))
    paper_labels = np.where(paper_labels < 0, cfg['K'] - 1, paper_labels)

    results[cfg['name']] = {
        'K': cfg['K'], 'n_nodes': int(n), 'n_pca_components': int(n_comp),
        'pca_explained_variance_ratio_sum': evr,
        'gamma': GAMMA,
        'overall_iia': edge_density(adj),
        'paper': evaluate_partition(adj, paper_labels, cfg['K']),
        'pca':   evaluate_partition(adj, pca_lab,     cfg['K']),
        'label_agreement': {
            'ARI': float(adjusted_rand_score(paper_labels, pca_lab)),
            'NMI': float(normalized_mutual_info_score(paper_labels, pca_lab)),
        },
    }
    r = results[cfg['name']]
    p_tgt = max(r['paper']['per_bucket'].values(), key=lambda b: b['iia'])
    print(f"  overall IIA={r['overall_iia']:.3f}  "
          f"paper target IIA={p_tgt['iia']:.3f} (size {p_tgt['size']})  "
          f"pca wIIA={r['pca']['weighted_mean_iia']:.3f}  "
          f"ARI={r['label_agreement']['ARI']:.3f}")

del _gpt2_model, _gpt2_tok
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Experiment 3 — Entity Binding @ L15 (Gemma-2-2B)

The author's `extract_activations_at_layer` helper depends on `causalab.tasks.entity_binding`, which is only in an internal fork. To stay self-contained, this notebook **reconstructs the prompt directly from the saved samples** (every sample carries its own `statement_template` plus all `entity_g*_e*` / `query_e*` fields) and extracts the layer-15 last-token residual stream with a raw HuggingFace forward hook.

**Caveat.** The question wording (the line between the statements and `Answer:`) is reconstructed from patterns in the `sample_action_config` shipped with public causalab; the exact phrasing used during DAS training may differ. The activations therefore won't be bit-identical to the author's, but for a *geometry vs interchange-graph* baseline this is fine — the comparison still measures how well activation-space clustering recovers the graph partition.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from collections import Counter

EB_NAME       = 'Entity Binding @ L15'
EB_MODEL      = 'google/gemma-2-2b-it'
EB_LAYER      = 15
EB_TASK       = 'filling_liquids'
EB_NUM_GROUPS = 10
EB_K          = 2
EB_TAG        = f'{EB_TASK}_{EB_LAYER}_{EB_NUM_GROUPS}_512_google_gemma-2-2b-it'
EB_ART        = os.path.join(REPO_ROOT, 'experiments', 'entity_binding', 'artifacts', 'partition_results')

# --- prompt builder (no causalab dependency) ---
# Matches MVCA / apple-bucket create_custom_config (run_das.py L170-176).
EB_PROMPT_PREFIX     = "We will ask a question about the following sentences. Only return the answer, no other text.\n\n"
EB_PROMPT_SUFFIX     = "\nAnswer:"
EB_STMT_QUESTION_SEP = "\n\n"

# Question templates keyed by (query_indices, answer_index). Patterned on `sample_action_config`
# in public causalab; the exact wording of the filling_liquids variant is in the author's internal
# causalab fork and could not be recovered, so this is a best-guess approximation.
EB_QUESTION_TEMPLATES = {
    ((0,), 1): "What does {q0} fill?",
    ((0,), 2): "What does {q0} fill the container with?",
    ((1,), 0): "Who fills the {q1}?",
    ((1,), 2): "What is the {q1} filled with?",
    ((2,), 0): "Who fills something with {q2}?",
    ((2,), 1): "What does someone fill with {q2}?",
    ((0, 1), 2): "What does {q0} fill the {q1} with?",
    ((0, 2), 1): "What does {q0} fill with {q2}?",
    ((1, 2), 0): "Who fills the {q1} with {q2}?",
}

def eb_build_prompt(sample):
    active = int(sample['active_groups'])
    tpl = sample['statement_template']
    statements = [tpl.format(e0=sample[f'entity_g{g}_e0'],
                             e1=sample[f'entity_g{g}_e1'],
                             e2=sample[f'entity_g{g}_e2']) for g in range(active)]
    if active >= 2:
        body = ', '.join(statements[:-1]) + ', and ' + statements[-1] + '.'
    else:
        body = statements[0] + '.'
    qi = tuple(sample['query_indices']) if not isinstance(sample['query_indices'], tuple) else sample['query_indices']
    ai = int(sample['answer_index'])
    q_tpl = EB_QUESTION_TEMPLATES.get((qi, ai))
    if q_tpl is None:
        q_tpl = "Answer the question about the sentences above."  # safe fallback
    question = q_tpl.format(q0=sample.get('query_e0', ''),
                            q1=sample.get('query_e1', ''),
                            q2=sample.get('query_e2', ''))
    return EB_PROMPT_PREFIX + body + EB_STMT_QUESTION_SEP + question + EB_PROMPT_SUFFIX

# --- data ---

with open(os.path.join(EB_ART, f'graph_{EB_TAG}.pkl'), 'rb') as f:
    eb_adj = np.asarray(pickle.load(f), dtype=bool)
with open(os.path.join(EB_ART, f'filtered_input_samples_{EB_TAG}.pkl'), 'rb') as f:
    eb_samples = pickle.load(f)
assert eb_adj.shape[0] == len(eb_samples)
eb_n = eb_adj.shape[0]

# Show what gamma the saved JSON used (for transparency).
with open(os.path.join(EB_ART, f'partition_results_{EB_TAG}.json')) as f:
    _eb_saved = json.load(f)
print(f"Saved JSON used gamma={_eb_saved.get('gamma')}; recomputing with gamma={GAMMA} to match paper.")

_qi_ai_counts = Counter((tuple(s['query_indices']), int(s['answer_index'])) for s in eb_samples)
print(f'(query_indices, answer_index) distribution across {eb_n} samples:')
for k, v in _qi_ai_counts.most_common():
    print(f'  {k}: {v}  -- using template: {EB_QUESTION_TEMPLATES.get(k, "[fallback]")!r}')

print('\n--- sample 0 reconstructed prompt ---')
print(eb_build_prompt(eb_samples[0]))
print('--- end ---\n')

# --- activation extraction (load cached if available) ---

eb_cache = os.path.join(EB_ART, f'activations_{EB_TAG}_L{EB_LAYER}_lastTok.pt')
if os.path.exists(eb_cache):
    print(f'Loading cached activations from {eb_cache}')
    eb_acts = torch.load(eb_cache, map_location='cpu')
else:
    print(f'Loading {EB_MODEL} ...')
    eb_tok = AutoTokenizer.from_pretrained(EB_MODEL)
    eb_tok.padding_side = 'left'
    if eb_tok.pad_token is None:
        eb_tok.pad_token = eb_tok.eos_token
    eb_model = AutoModelForCausalLM.from_pretrained(
        EB_MODEL, torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32, device_map=DEVICE,
    ).eval()

    acts_buf = [None]
    def _hook(module, inp, out):
        acts_buf[0] = (out[0] if isinstance(out, tuple) else out).detach()
    hook = eb_model.model.layers[EB_LAYER].register_forward_hook(_hook)

    eb_acts_list = []
    with torch.no_grad():
        for s in tqdm(eb_samples, desc=f'extract L{EB_LAYER} last-token'):
            prompt = eb_build_prompt(s)
            enc = eb_tok(prompt, return_tensors='pt').to(DEVICE)
            eb_model(**enc)
            eb_acts_list.append(acts_buf[0][0, -1, :].float().cpu())
    hook.remove()
    eb_acts = torch.stack(eb_acts_list)
    torch.save(eb_acts, eb_cache)
    del eb_model, eb_tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
X_eb = eb_acts.detach().cpu().numpy().astype(np.float32)
print('Activations shape:', X_eb.shape)

# --- PCA + KMeans + paper-method recompute + comparison ---

eb_pca_lab, eb_evr, eb_ncomp = pca_kmeans_cluster(X_eb, EB_K, N_PCA_COMPONENTS, SEED)
eb_paper_labels = np.asarray(quasi_clique_partition(eb_adj, EB_K, GAMMA, MIN_CLIQUE_SIZE))
eb_paper_labels = np.where(eb_paper_labels < 0, EB_K - 1, eb_paper_labels)
assert len(eb_paper_labels) == eb_n

results[EB_NAME] = {
    'K': EB_K, 'n_nodes': int(eb_n), 'n_pca_components': int(eb_ncomp),
    'pca_explained_variance_ratio_sum': eb_evr,
    'gamma': GAMMA,
    'overall_iia': edge_density(eb_adj),
    'paper': evaluate_partition(eb_adj, eb_paper_labels, EB_K),
    'pca':   evaluate_partition(eb_adj, eb_pca_lab,     EB_K),
    'label_agreement': {
        'ARI': float(adjusted_rand_score(eb_paper_labels, eb_pca_lab)),
        'NMI': float(normalized_mutual_info_score(eb_paper_labels, eb_pca_lab)),
    },
}
r = results[EB_NAME]
p_tgt = max(r['paper']['per_bucket'].values(), key=lambda b: b['iia'])
print(f"{EB_NAME}: overall IIA={r['overall_iia']:.3f}  "
      f"paper target IIA={p_tgt['iia']:.3f} (size {p_tgt['size']})  "
      f"pca wIIA={r['pca']['weighted_mean_iia']:.3f}  "
      f"ARI={r['label_agreement']['ARI']:.3f}")

## 5. Experiment 4 — RAVEL Language @ L14 (Llama-3.1-8B)

Uses the layer-14 activations already cached in the repo (`activations_layer14_n200.pt`), so no Llama load is needed. The paper's quasi-clique partition isn't saved as JSON for factual recall; we recompute it inline from `graph.pkl` with the same `quasi_clique_partition` helper the paper uses in `step4_classifier.py`.

In [ ]:
FR_NAME           = 'RAVEL Language @ L14'
FR_LAYER          = 14
FR_K              = 2
FR_ART            = os.path.join(REPO_ROOT, 'experiments', 'factual_recall', 'artifacts')
FR_TEST_DIR       = os.path.join(FR_ART, 'test_results_das')
FR_CLF_DIR        = os.path.join(FR_ART, 'classifier_results')

with open(os.path.join(FR_TEST_DIR, 'graph.pkl'), 'rb') as f:
    fr_adj_d = np.asarray(pickle.load(f), dtype=bool)
fr_adj = fr_adj_d & fr_adj_d.T
fr_n = fr_adj.shape[0]

fr_acts_path = os.path.join(FR_CLF_DIR, f'activations_layer{FR_LAYER}_n{fr_n}.pt')
assert os.path.exists(fr_acts_path), (
    f'Cached activations not found at {fr_acts_path}. '
    'Re-extract by running scripts/step4_classifier.py once locally.'
)
fr_acts = torch.load(fr_acts_path, map_location='cpu')
X_fr = fr_acts.detach().cpu().numpy().astype(np.float32)
assert X_fr.shape[0] == fr_n

fr_pca_lab, fr_evr, fr_ncomp = pca_kmeans_cluster(X_fr, FR_K, N_PCA_COMPONENTS, SEED)
fr_paper_labels = np.asarray(quasi_clique_partition(fr_adj, FR_K, GAMMA, MIN_CLIQUE_SIZE))
fr_paper_labels = np.where(fr_paper_labels < 0, FR_K - 1, fr_paper_labels)

results[FR_NAME] = {
    'K': FR_K, 'n_nodes': int(fr_n), 'n_pca_components': int(fr_ncomp),
    'pca_explained_variance_ratio_sum': fr_evr,
    'gamma': GAMMA,
    'overall_iia': edge_density(fr_adj),
    'paper': evaluate_partition(fr_adj, fr_paper_labels, FR_K),
    'pca':   evaluate_partition(fr_adj, fr_pca_lab,     FR_K),
    'label_agreement': {
        'ARI': float(adjusted_rand_score(fr_paper_labels, fr_pca_lab)),
        'NMI': float(normalized_mutual_info_score(fr_paper_labels, fr_pca_lab)),
    },
}
r = results[FR_NAME]
p_tgt = max(r['paper']['per_bucket'].values(), key=lambda b: b['iia'])
print(f"{FR_NAME}: overall IIA={r['overall_iia']:.3f}  "
      f"paper target IIA={p_tgt['iia']:.3f} (size {p_tgt['size']})  "
      f"pca wIIA={r['pca']['weighted_mean_iia']:.3f}  "
      f"ARI={r['label_agreement']['ARI']:.3f}")

## 6. Summary table

Side-by-side comparison of our quasi-clique partition vs the PCA + K-means baseline. For each method we sort the K buckets by descending IIA and call the top one **Target** (the high-density bucket — analogous to a quasi-clique) and the next one **Other** (the residual bucket). With `K=2` this is unambiguous.

Columns: `Target size / Target IIA / Other size / Other IIA`, the graph's `Overall IIA` (full-graph edge density), and `ARI / NMI` of the PCA partition against ours. (We intentionally omit weighted-mean IIA — with one dense and one sparse bucket it is dominated by the large residual bucket and is not informative.)

In [ ]:
import pandas as pd
from IPython.display import display

def split_target_other(per_bucket):
    s = sorted(per_bucket.values(), key=lambda b: -b['iia'])
    return s[0], s[1]

rows = []
for name, r in results.items():
    overall = r['overall_iia']
    p_tgt, p_oth = split_target_other(r['paper']['per_bucket'])
    q_tgt, q_oth = split_target_other(r['pca']['per_bucket'])
    rows.append({
        'Experiment': name, 'Method': 'Ours (quasi-clique)',
        'Target size': p_tgt['size'], 'Target IIA': round(p_tgt['iia'], 3),
        'Other size':  p_oth['size'], 'Other IIA':  round(p_oth['iia'], 3),
        'Overall IIA': round(overall, 3),
        'ARI vs ours': '—', 'NMI vs ours': '—',
    })
    rows.append({
        'Experiment': name, 'Method': 'PCA baseline',
        'Target size': q_tgt['size'], 'Target IIA': round(q_tgt['iia'], 3),
        'Other size':  q_oth['size'], 'Other IIA':  round(q_oth['iia'], 3),
        'Overall IIA': round(overall, 3),
        'ARI vs ours': round(r['label_agreement']['ARI'], 3),
        'NMI vs ours': round(r['label_agreement']['NMI'], 3),
    })

df = pd.DataFrame(rows)
display(df)

### 6.1 Rendered comparison figure

Renders the same comparison as a publication-style table image (`notebooks/pca_baseline_table.png`) suitable for dropping into the rebuttal. Weighted-mean IIA is omitted; the **Target IIA** column is the key number.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display as idisplay

col_labels = ['Experiment', 'Method', 'Target\nsize', 'Target\nIIA',
              'Other\nsize', 'Other\nIIA', 'Overall\nIIA', 'ARI', 'NMI']
fig_rows, row_meta = [], []
for ei, (name, r) in enumerate(results.items()):
    overall = r['overall_iia']
    p_t, p_o = split_target_other(r['paper']['per_bucket'])
    q_t, q_o = split_target_other(r['pca']['per_bucket'])
    fig_rows.append([name, 'Ours (quasi-clique)', p_t['size'], f"{p_t['iia']:.3f}",
                     p_o['size'], f"{p_o['iia']:.3f}", f"{overall:.3f}", '—', '—'])
    row_meta.append(ei)
    fig_rows.append(['', 'PCA baseline', q_t['size'], f"{q_t['iia']:.3f}",
                     q_o['size'], f"{q_o['iia']:.3f}", f"{overall:.3f}",
                     f"{r['label_agreement']['ARI']:.3f}", f"{r['label_agreement']['NMI']:.3f}"])
    row_meta.append(ei)

fig, ax = plt.subplots(figsize=(13, 0.62 * (len(fig_rows) + 2)))
ax.axis('off')
ax.set_title('PCA-clustering baseline vs. our interchangeability-graph partition\n'
             r'(quasi-clique density threshold $\gamma=0.98$, K=2, PCA=50 dims, seed=42)',
             fontsize=13, fontweight='bold', pad=18)

tbl = ax.table(cellText=fig_rows, colLabels=col_labels, cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10.5); tbl.scale(1, 1.65)
group_alt = ['#ffffff', '#f4f4f4']
for (rr, cc), cell in tbl.get_celld().items():
    cell.set_edgecolor('#b0b0b0'); cell.set_linewidth(0.6)
    if rr == 0:
        cell.set_facecolor('#cfe2f3'); cell.set_text_props(fontweight='bold')
        cell.set_height(cell.get_height() * 1.25); continue
    cell.set_facecolor(group_alt[row_meta[rr - 1] % 2])
    if cc in (0, 1):
        cell.set_text_props(ha='left'); cell._loc = 'left'
    if cc == 3:
        cell.set_text_props(fontweight='bold')
for (rr, cc), cell in tbl.get_celld().items():
    if cc in (0, 1):
        cell.set_width({0: 0.20, 1: 0.17}[cc])

fig.text(0.5, 0.02, 'Target = densest bucket (high-IIA region); Other = remainder. '
         'ARI / NMI measure agreement of the PCA partition with ours '
         '(higher = more similar; near 0 = unrelated).',
         ha='center', va='bottom', fontsize=8.7, style='italic', wrap=True)

png_path = os.path.join(OUT_DIR if 'OUT_DIR' in globals() else os.path.join(REPO_ROOT, 'notebooks'),
                        'pca_baseline_table.png')
os.makedirs(os.path.dirname(png_path), exist_ok=True)
fig.savefig(png_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig)
print('Saved:', png_path)
idisplay(Image(png_path))

## 7. Persist results

Saves the comparison as both CSV (for pasting into the rebuttal / paper) and a JSON dump (full per-bucket detail + cluster labels) at the repo root.

In [ ]:
OUT_DIR = os.path.join(REPO_ROOT, 'notebooks')
os.makedirs(OUT_DIR, exist_ok=True)
csv_path  = os.path.join(OUT_DIR, 'pca_baseline_comparison.csv')
json_path = os.path.join(OUT_DIR, 'pca_baseline_comparison.json')

df.to_csv(csv_path, index=False)
with open(json_path, 'w') as f:
    json.dump({'seed': SEED, 'n_pca_components': N_PCA_COMPONENTS, 'results': results}, f, indent=2)

print('Saved:', csv_path)
print('Saved:', json_path)